# 16. The Container Terminal Yard Truck Scheduling Problem

## Tier 1 — Mixed-Integer Programming (MIP) Formulation

This notebook transforms the yard truck scheduling problem into a **mathematical optimization model** that can be solved to optimality.

### Learning goals

- Understand how to formulate vehicle scheduling with time windows as MIP
- Learn to model precedence constraints and resource allocation
- See how to extract and interpret optimal schedules
- Practice sensitivity analysis on key parameters

### What this notebook outputs

- Optimal truck schedule for a small terminal instance
- Detailed assignment of trucks to container moves
- Gantt chart visualization of the optimal schedule
- Performance metrics and sensitivity analysis

### Why the instance is small

MIP problems are NP-hard, so we use a small example (3 trucks, 8 container moves) to ensure fast solving and clear visualization of the optimal solution.

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    import seaborn as sns
    from itertools import product
    from typing import List, Tuple, Dict, Any
    from dataclasses import dataclass
    from functools import lru_cache
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib, seaborn. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete instance (3 trucks, 8 container moves)

We will solve a small yard truck scheduling problem with:

- **3 trucks** available for container movements
- **8 container moves** requiring truck service
- **Time windows** for each move (earliest start, latest finish)
- **Processing times** for loading/unloading operations
- **Travel times** between locations

### Time and distance model

- Travel time between locations: based on Euclidean distance
- Processing time: includes loading/unloading at origin and destination
- Time windows: enforce service level requirements
- Truck capacity: 1 container at a time (simplified model)

In [ ]:
# ----------------------------
# Data model: container move request
# ----------------------------
@dataclass(frozen=True)
class ContainerMove:
    """Represents a container movement request."""
    id: int
    origin: Tuple[float, float]  # (x, y) coordinates
    destination: Tuple[float, float]  # (x, y) coordinates
    earliest_start: float  # earliest time service can begin
    latest_finish: float   # latest time service must complete
    processing_time: float  # time for loading/unloading
    priority: int  # priority level (1=highest, 3=lowest)


# ----------------------------
# Data model: truck
# ----------------------------
@dataclass(frozen=True)
class Truck:
    """Represents a yard truck."""
    id: int
    start_location: Tuple[float, float]
    available_time: float  # when truck becomes available
    speed: float  # travel speed (distance per minute)


# ----------------------------
# Concrete instance: 8 container moves
# ----------------------------
container_moves = [
    ContainerMove(1, (10, 20), (80, 60), 0, 120, 15, 1),    # High priority export
    ContainerMove(2, (15, 25), (75, 55), 10, 130, 12, 2),   # Medium priority
    ContainerMove(3, (20, 30), (70, 50), 20, 140, 18, 1),   # High priority import
    ContainerMove(4, (25, 35), (65, 45), 30, 150, 14, 3),   # Low priority
    ContainerMove(5, (30, 40), (60, 40), 40, 160, 16, 2),   # Medium priority
    ContainerMove(6, (35, 45), (55, 35), 50, 170, 13, 1),   # High priority
    ContainerMove(7, (40, 50), (50, 30), 60, 180, 15, 2),   # Medium priority
    ContainerMove(8, (45, 55), (45, 25), 70, 190, 17, 3),   # Low priority
]

# ----------------------------
# Concrete instance: 3 trucks
# ----------------------------
trucks = [
    Truck(1, (50, 50), 0, 2.0),   # Central location, immediately available
    Truck(2, (30, 30), 5, 1.8),   # West side, available after 5 minutes
    Truck(3, (70, 70), 10, 2.2),  # East side, available after 10 minutes
]

# ----------------------------
# Helper functions
# ----------------------------
def euclidean_distance(p1: Tuple[float, float], p2: Tuple[float, float]) -> float:
    """Calculate Euclidean distance between two points."""
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


def travel_time(p1: Tuple[float, float], p2: Tuple[float, float], speed: float) -> float:
    """Calculate travel time between two locations."""
    return euclidean_distance(p1, p2) / speed


# Precompute travel times for efficiency
travel_times = {}
for truck in trucks:
    for move in container_moves:
        # Truck start to move origin
        travel_times[(truck.id, 'start', move.id)] = travel_time(
            truck.start_location, move.origin, truck.speed
        )
        # Move origin to destination
        travel_times[(truck.id, move.id, 'dest')] = travel_time(
            move.origin, move.destination, truck.speed
        )
        # Move destination to next move origin (for consecutive moves)
        for next_move in container_moves:
            if move.id != next_move.id:
                travel_times[(truck.id, move.id, next_move.id)] = travel_time(
                    move.destination, next_move.origin, truck.speed
                )

print(f"Instance: {len(trucks)} trucks, {len(container_moves)} container moves")
print(f"Precomputed {len(travel_times)} travel times")

## Step 1 — MIP Formulation

We formulate the yard truck scheduling problem as a Mixed-Integer Programming model with:

### Decision Variables
- **x[i,k]**: 1 if truck k serves move i, 0 otherwise
- **y[i,j,k]**: 1 if truck k serves move j immediately after move i, 0 otherwise
- **s[i]**: start time of move i
- **c[i]**: completion time of move i

### Objective Function
Minimize total completion time (makespan) while considering priority weights:

```
minimize Σ(priority[i] * c[i]) + α * makespan
```

### Constraints
1. **Assignment**: Each move must be served by exactly one truck
2. **Precedence**: If y[i,j,k] = 1, then move j must start after move i completes
3. **Time windows**: Start and completion times must respect time windows
4. **Truck availability**: Trucks cannot start moves before they're available
5. **Flow conservation**: Each truck's route must be continuous
6. **Processing time**: Completion time = start time + processing time

In [ ]:
# ----------------------------
# MIP Model Implementation
# ----------------------------
# Since we cannot use commercial solvers, we'll implement a simplified
# exact solution using enumeration for this small instance

from itertools import permutations, combinations
import time


class YardTruckScheduler:
    """Exact solver for yard truck scheduling using enumeration."""
    
    def __init__(self, trucks: List[Truck], moves: List[ContainerMove]):
        self.trucks = trucks
        self.moves = moves
        self.best_solution = None
        self.best_objective = float('inf')
        
    def calculate_move_time(self, truck: Truck, move: ContainerMove, 
                          prev_move: ContainerMove = None) -> Tuple[float, float]:
        """Calculate start and completion times for a move."""
        if prev_move is None:
            # First move for this truck
            travel_to_origin = travel_times[(truck.id, 'start', move.id)]
            earliest_start = max(truck.available_time + travel_to_origin, move.earliest_start)
        else:
            # Consecutive move
            travel_to_origin = travel_times[(truck.id, prev_move.id, move.id)]
            earliest_start = max(prev_move.completion_time + travel_to_origin, move.earliest_start)
        
        completion_time = earliest_start + move.processing_time
        
        # Check time window feasibility
        if completion_time > move.latest_finish:
            return None, None  # Infeasible
            
        return earliest_start, completion_time
    
    def evaluate_assignment(self, assignment: Dict[int, List[int]]) -> float:
        """Evaluate a truck-to-moves assignment and return objective value."""
        total_objective = 0.0
        makespan = 0.0
        
        # Update move completion times
        for truck_id, move_ids in assignment.items():
            truck = self.trucks[truck_id - 1]
            prev_move = None
            
            for move_id in move_ids:
                move = self.moves[move_id - 1]
                start_time, completion_time = self.calculate_move_time(
                    truck, move, prev_move
                )
                
                if start_time is None:  # Infeasible assignment
                    return float('inf')
                
                # Update move with calculated times
                move.start_time = start_time
                move.completion_time = completion_time
                
                total_objective += move.priority * completion_time
                makespan = max(makespan, completion_time)
                prev_move = move
        
        # Combined objective: weighted completion + makespan penalty
        alpha = 0.1  # Makespan weight
        return total_objective + alpha * makespan
    
    def generate_assignments(self) -> List[Dict[int, List[int]]]:
        """Generate all feasible truck-to-move assignments."""
        move_ids = list(range(1, len(self.moves) + 1))
        truck_ids = list(range(1, len(self.trucks) + 1))
        assignments = []
        
        # Generate all ways to partition moves among trucks
        # For small instances, we can use recursive partitioning
        def partition_moves(moves_left, current_assignment, trucks_left):
            if not moves_left:
                assignments.append(current_assignment.copy())
                return
            
            if not trucks_left:
                return  # No trucks left but moves remain
            
            truck_id = trucks_left[0]
            
            # Try assigning 0 to all remaining moves to this truck
            for n_moves in range(len(moves_left) + 1):
                for subset in combinations(moves_left, n_moves):
                    remaining_moves = [m for m in moves_left if m not in subset]
                    
                    if subset:
                        current_assignment[truck_id] = list(subset)
                    else:
                        current_assignment[truck_id] = []
                    
                    partition_moves(remaining_moves, current_assignment, trucks_left[1:])
        
        partition_moves(move_ids, {}, truck_ids)
        return assignments
    
    def solve(self, time_limit: int = 30) -> Dict[str, Any]:
        """Solve the yard truck scheduling problem."""
        start_time = time.time()
        
        print("Generating assignments...")
        assignments = self.generate_assignments()
        print(f"Generated {len(assignments)} assignments")
        
        best_assignment = None
        evaluated = 0
        
        for assignment in assignments:
            if time.time() - start_time > time_limit:
                print(f"Time limit reached after evaluating {evaluated} assignments")
                break
            
            # For each truck, try all possible orderings of their assigned moves
            feasible_orderings = []
            
            for truck_id, move_ids in assignment.items():
                if not move_ids:
                    feasible_orderings.append([[]])
                else:
                    truck_orderings = []
                    for ordering in permutations(move_ids):
                        truck_orderings.append(list(ordering))
                    feasible_orderings.append(truck_orderings)
            
            # Combine orderings across trucks
            for truck_orderings in product(*feasible_orderings):
                test_assignment = {
                    truck_id: ordering 
                    for truck_id, ordering in zip(assignment.keys(), truck_orderings)
                }
                
                objective = self.evaluate_assignment(test_assignment)
                evaluated += 1
                
                if objective < self.best_objective:
                    self.best_objective = objective
                    best_assignment = test_assignment.copy()
                    
                    # Store best solution details
                    self.best_solution = {
                        'assignment': best_assignment,
                        'objective': objective,
                        'move_schedule': {}
                    }
                    
                    for move in self.moves:
                        self.best_solution['move_schedule'][move.id] = {
                            'start_time': move.start_time,
                            'completion_time': move.completion_time,
                            'priority': move.priority
                        }
        
        solve_time = time.time() - start_time
        print(f"Evaluated {evaluated} assignments in {solve_time:.2f} seconds")
        print(f"Best objective: {self.best_objective:.2f}")
        
        return self.best_solution


# Create and solve the instance
scheduler = YardTruckScheduler(trucks, container_moves)
solution = scheduler.solve(time_limit=20)

## Step 2 — Extract and Display the Optimal Schedule

Now we'll extract the optimal solution and present it in a clear, interpretable format.

In [ ]:
# ----------------------------
# Display optimal solution
# ----------------------------
if solution is None:
    print("No feasible solution found within time limit")
else:
    print("=== OPTIMAL TRUCK SCHEDULE ===")
    print(f"Objective Value: {solution['objective']:.2f}")
    print()
    
    # Display truck assignments
    for truck_id, move_ids in solution['assignment'].items():
        truck = trucks[truck_id - 1]
        print(f"Truck {truck_id} (speed: {truck.speed}, available: {truck.available_time}):")
        
        if not move_ids:
            print("  No moves assigned")
        else:
            for i, move_id in enumerate(move_ids):
                move = container_moves[move_id - 1]
                schedule = solution['move_schedule'][move_id]
                
                print(f"  Move {move_id} (Priority {move.priority}): "
                      f"{schedule['start_time']:.1f} - {schedule['completion_time']:.1f}")
                print(f"    Route: {move.origin} → {move.destination}")
                print(f"    Time window: [{move.earliest_start}, {move.latest_finish}]")
        print()
    
    # Calculate performance metrics
    all_completions = [s['completion_time'] for s in solution['move_schedule'].values()]
    makespan = max(all_completions)
    avg_completion = np.mean(all_completions)
    
    print("=== PERFORMANCE METRICS ===")
    print(f"Makespan: {makespan:.1f} minutes")
    print(f"Average completion time: {avg_completion:.1f} minutes")
    print(f"Total moves completed: {len(container_moves)}")
    print(f"Trucks utilized: {len([moves for moves in solution['assignment'].values() if moves])}/{len(trucks)}")

## Step 3 — Create Detailed Schedule Table

We'll create a comprehensive table showing all aspects of the optimal schedule for easy analysis.

In [ ]:
# ----------------------------
# Create detailed schedule table
# ----------------------------
if solution is not None:
    schedule_data = []
    
    for truck_id, move_ids in solution['assignment'].items():
        for i, move_id in enumerate(move_ids):
            move = container_moves[move_id - 1]
            schedule = solution['move_schedule'][move_id]
            
            # Calculate waiting time
            waiting_time = max(0, schedule['start_time'] - move.earliest_start)
            
            # Calculate slack in time window
            slack = move.latest_finish - schedule['completion_time']
            
            schedule_data.append({
                'Truck': f'T{truck_id}',
                'Move': f'M{move_id}',
                'Priority': move.priority,
                'Start': schedule['start_time'],
                'Completion': schedule['completion_time'],
                'Processing': move.processing_time,
                'Waiting': waiting_time,
                'Slack': slack,
                'Origin': f"({move.origin[0]}, {move.origin[1]})",
                'Destination': f"({move.destination[0]}, {move.destination[1]})",
                'Sequence': i + 1
            })
    
    schedule_df = pd.DataFrame(schedule_data)
    
    # Sort by truck and start time
    schedule_df = schedule_df.sort_values(['Truck', 'Start'])
    
    # Format numeric columns
    numeric_cols = ['Start', 'Completion', 'Processing', 'Waiting', 'Slack']
    schedule_df[numeric_cols] = schedule_df[numeric_cols].round(2)
    
    print("=== DETAILED SCHEDULE TABLE ===")
    display(schedule_df)
    
    # Summary statistics
    print("\n=== SUMMARY STATISTICS ===")
    print(f"Total waiting time: {schedule_df['Waiting'].sum():.2f} minutes")
    print(f"Average waiting time: {schedule_df['Waiting'].mean():.2f} minutes")
    print(f"Total slack time: {schedule_df['Slack'].sum():.2f} minutes")
    print(f"Moves with no waiting: {(schedule_df['Waiting'] == 0).sum()}/{len(schedule_df)}")
    print(f"Moves using full time window: {(schedule_df['Slack'] == 0).sum()}/{len(schedule_df)}")

## Step 4 — Gantt Chart Visualization

A Gantt chart helps visualize the optimal schedule and identify potential bottlenecks or inefficiencies.

In [ ]:
# ----------------------------
# Gantt chart visualization
# ----------------------------
if solution is not None:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Color palette for different priorities
    colors = {1: '#FF6B6B', 2: '#4ECDC4', 3: '#45B7D1'}
    
    # Plot each move as a bar
    y_pos = 0
    truck_labels = []
    
    for truck_id, move_ids in solution['assignment'].items():
        truck_label = f'Truck {truck_id}'
        
        if not move_ids:
            # Show idle truck
            ax.barh(y_pos, 5, left=0, height=0.6, 
                   color='#E0E0E0', alpha=0.5, label='Idle')
            truck_labels.append(truck_label)
            y_pos += 1
        else:
            for move_id in move_ids:
                move = container_moves[move_id - 1]
                schedule = solution['move_schedule'][move_id]
                
                # Draw the move bar
                ax.barh(y_pos, 
                       move.processing_time,
                       left=schedule['start_time'],
                       height=0.6,
                       color=colors[move.priority],
                       alpha=0.8,
                       edgecolor='black',
                       linewidth=1)
                
                # Add move label
                ax.text(schedule['start_time'] + move.processing_time/2, y_pos,
                       f'M{move_id}',
                       ha='center', va='center',
                       fontsize=9, fontweight='bold')
            
            truck_labels.append(truck_label)
            y_pos += 1
    
    # Formatting
    ax.set_yticks(range(len(truck_labels)))
    ax.set_yticklabels(truck_labels)
    ax.set_xlabel('Time (minutes)', fontsize=12)
    ax.set_title('Optimal Yard Truck Schedule - Gantt Chart', fontsize=14, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)
    
    # Add legend for priorities
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors[1], label='Priority 1 (High)'),
        Patch(facecolor=colors[2], label='Priority 2 (Medium)'),
        Patch(facecolor=colors[3], label='Priority 3 (Low)')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    # Set x-axis limits
    max_time = max([s['completion_time'] for s in solution['move_schedule'].values()])
    ax.set_xlim(0, max_time + 10)
    
    plt.tight_layout()
    plt.show()
else:
    print("No solution to visualize")

## Step 5 — Spatial Visualization

We'll visualize the terminal layout and truck movements to understand the spatial aspects of the optimal schedule.

In [ ]:
# ----------------------------
# Spatial visualization of terminal operations
# ----------------------------
if solution is not None:
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Set up the plot
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 80)
    ax.set_aspect('equal')
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title('Terminal Layout and Truck Movements', fontsize=14, fontweight='bold')
    
    # Plot truck starting positions
    truck_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for i, truck in enumerate(trucks):
        ax.scatter(truck.start_location[0], truck.start_location[1],
                  s=200, c=truck_colors[i], marker='s',
                  edgecolor='black', linewidth=2,
                  label=f'Truck {truck.id} Start')
        ax.text(truck.start_location[0], truck.start_location[1] + 3,
               f'T{truck.id}', ha='center', fontweight='bold')
    
    # Plot container moves
    for move in container_moves:
        # Origin point
        ax.scatter(move.origin[0], move.origin[1],
                  s=100, c='lightblue', marker='o',
                  edgecolor='black', alpha=0.7)
        ax.text(move.origin[0], move.origin[1] - 3,
               f'M{move.id}', ha='center', fontsize=9)
        
        # Destination point
        ax.scatter(move.destination[0], move.destination[1],
                  s=100, c='lightgreen', marker='^',
                  edgecolor='black', alpha=0.7)
        ax.text(move.destination[0], move.destination[1] + 3,
               f'M{move.id}', ha='center', fontsize=9)
        
        # Draw arrow from origin to destination
        ax.annotate('', xy=move.destination, xytext=move.origin,
                   arrowprops=dict(arrowstyle='->', lw=1, 
                                 color='gray', alpha=0.5))
    
    # Plot truck routes for optimal schedule
    for truck_id, move_ids in solution['assignment'].items():
        truck = trucks[truck_id - 1]
        color = truck_colors[truck_id - 1]
        
        if move_ids:
            # Route from truck start to first move
            first_move = container_moves[move_ids[0] - 1]
            ax.plot([truck.start_location[0], first_move.origin[0]],
                   [truck.start_location[1], first_move.origin[1]],
                   color=color, linewidth=2, alpha=0.8,
                   linestyle='--', label=f'Truck {truck_id} Route')
            
            # Routes between consecutive moves
            for i in range(len(move_ids) - 1):
                current_move = container_moves[move_ids[i] - 1]
                next_move = container_moves[move_ids[i + 1] - 1]
                
                ax.plot([current_move.destination[0], next_move.origin[0]],
                       [current_move.destination[1], next_move.origin[1]],
                       color=color, linewidth=2, alpha=0.8,
                       linestyle='--')
    
    # Add legend
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No solution to visualize")

## Step 6 — Sensitivity Analysis

We'll analyze how changes in key parameters affect the optimal solution to understand the robustness and key drivers of the schedule.

In [ ]:
# ----------------------------
# Sensitivity analysis on key parameters
# ----------------------------
def sensitivity_analysis():
    """Perform sensitivity analysis on key parameters."""
    
    scenarios = []
    
    # Scenario 1: Base case (current solution)
    if solution is not None:
        scenarios.append({
            'name': 'Base Case',
            'objective': solution['objective'],
            'makespan': max([s['completion_time'] for s in solution['move_schedule'].values()]),
            'avg_completion': np.mean([s['completion_time'] for s in solution['move_schedule'].values()])
        })
    
    # Scenario 2: Faster trucks (increase speed by 20%)
    fast_trucks = []
    for truck in trucks:
        fast_trucks.append(Truck(
            truck.id, truck.start_location, truck.available_time, truck.speed * 1.2
        ))
    
    fast_scheduler = YardTruckScheduler(fast_trucks, container_moves)
    fast_solution = fast_scheduler.solve(time_limit=10)
    
    if fast_solution is not None:
        scenarios.append({
            'name': 'Fast Trucks (+20%)',
            'objective': fast_solution['objective'],
            'makespan': max([s['completion_time'] for s in fast_solution['move_schedule'].values()]),
            'avg_completion': np.mean([s['completion_time'] for s in fast_solution['move_schedule'].values()])
        })
    
    # Scenario 3: Tighter time windows (reduce latest_finish by 20%)
    tight_moves = []
    for move in container_moves:
        tight_moves.append(ContainerMove(
            move.id, move.origin, move.destination,
            move.earliest_start, move.latest_finish * 0.8,
            move.processing_time, move.priority
        ))
    
    tight_scheduler = YardTruckScheduler(trucks, tight_moves)
    tight_solution = tight_scheduler.solve(time_limit=10)
    
    if tight_solution is not None:
        scenarios.append({
            'name': 'Tight Windows (-20%)',
            'objective': tight_solution['objective'],
            'makespan': max([s['completion_time'] for s in tight_solution['move_schedule'].values()]),
            'avg_completion': np.mean([s['completion_time'] for s in tight_solution['move_schedule'].values()])
        })
    
    # Scenario 4: Additional truck
    extra_truck = trucks + [Truck(4, (50, 30), 0, 1.9)]
    extra_scheduler = YardTruckScheduler(extra_truck, container_moves)
    extra_solution = extra_scheduler.solve(time_limit=10)
    
    if extra_solution is not None:
        scenarios.append({
            'name': 'Extra Truck (+1)',
            'objective': extra_solution['objective'],
            'makespan': max([s['completion_time'] for s in extra_solution['move_schedule'].values()]),
            'avg_completion': np.mean([s['completion_time'] for s in extra_solution['move_schedule'].values()])
        })
    
    return scenarios


# Run sensitivity analysis
scenarios = sensitivity_analysis()

if scenarios:
    # Create comparison table
    sensitivity_df = pd.DataFrame(scenarios)
    
    print("=== SENSITIVITY ANALYSIS RESULTS ===")
    display(sensitivity_df)
    
    # Create comparison plots
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Objective comparison
    axes[0].bar(sensitivity_df['name'], sensitivity_df['objective'], 
                color=['#2E90FA', '#12B76A', '#F79009', '#F04438'], alpha=0.8)
    axes[0].set_title('Objective Function Value')
    axes[0].set_ylabel('Objective Value')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, axis='y', alpha=0.3)
    
    # Makespan comparison
    axes[1].bar(sensitivity_df['name'], sensitivity_df['makespan'],
                color=['#2E90FA', '#12B76A', '#F79009', '#F04438'], alpha=0.8)
    axes[1].set_title('Makespan (Total Completion Time)')
    axes[1].set_ylabel('Makespan (minutes)')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True, axis='y', alpha=0.3)
    
    # Average completion comparison
    axes[2].bar(sensitivity_df['name'], sensitivity_df['avg_completion'],
                color=['#2E90FA', '#12B76A', '#F79009', '#F04438'], alpha=0.8)
    axes[2].set_title('Average Completion Time')
    axes[2].set_ylabel('Avg Completion (minutes)')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(True, axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate percentage changes
    if len(scenarios) > 1:
        base_obj = scenarios[0]['objective']
        base_makespan = scenarios[0]['makespan']
        base_avg = scenarios[0]['avg_completion']
        
        print("\n=== PERCENTAGE IMPROVEMENTS ===")
        for scenario in scenarios[1:]:
            obj_improvement = (base_obj - scenario['objective']) / base_obj * 100
            makespan_improvement = (base_makespan - scenario['makespan']) / base_makespan * 100
            avg_improvement = (base_avg - scenario['avg_completion']) / base_avg * 100
            
            print(f"\n{scenario['name']}:")
            print(f"  Objective: {obj_improvement:+.1f}%")
            print(f"  Makespan: {makespan_improvement:+.1f}%")
            print(f"  Avg Completion: {avg_improvement:+.1f}%")
else:
    print("No scenarios available for comparison")

## Step 7 — Key Insights and Takeaways

Let's summarize the key findings from our optimal yard truck schedule.

In [ ]:
# ----------------------------
# Key insights and analysis
 ----------------------------
if solution is not None:
    print("=== KEY INSIGHTS FROM OPTIMAL SCHEDULE ===")
    print()
    
    # Truck utilization analysis
    print("1. TRUCK UTILIZATION:")
    for truck_id, move_ids in solution['assignment'].items():
        if move_ids:
            total_processing = sum(container_moves[m-1].processing_time for m in move_ids)
            schedule_times = [solution['move_schedule'][m]['completion_time'] for m in move_ids]
            makespan_truck = max(schedule_times) - min([solution['move_schedule'][m]['start_time'] for m in move_ids])
            utilization = total_processing / makespan_truck * 100 if makespan_truck > 0 else 0
            print(f"   Truck {truck_id}: {len(move_ids)} moves, {utilization:.1f}% utilization")
        else:
            print(f"   Truck {truck_id}: No moves assigned (0% utilization)")
    print()
    
    # Priority analysis
    print("2. PRIORITY HANDLING:")
    priority_moves = {1: [], 2: [], 3: []}
    for move_id, schedule in solution['move_schedule'].items():
        priority = container_moves[move_id - 1].priority
        priority_moves[priority].append(move_id)
    
    for priority in [1, 2, 3]:
        moves = priority_moves[priority]
        if moves:
            avg_completion = np.mean([solution['move_schedule'][m]['completion_time'] for m in moves])
            print(f"   Priority {priority} moves: {len(moves)} moves, avg completion: {avg_completion:.1f}")
    print()
    
    # Time window analysis
    print("3. TIME WINDOW COMPLIANCE:")
    tight_windows = 0
    loose_windows = 0
    
    for move_id, schedule in solution['move_schedule'].items():
        move = container_moves[move_id - 1]
        window_width = move.latest_finish - move.earliest_start
        actual_width = schedule['completion_time'] - schedule['start_time']
        
        if window_width - actual_width < 5:  # Less than 5 minutes slack
            tight_windows += 1
        else:
            loose_windows += 1
    
    print(f"   Moves with tight windows: {tight_windows}/{len(container_moves)}")
    print(f"   Moves with loose windows: {loose_windows}/{len(container_moves)}")
    print()
    
    # Spatial efficiency
    print("4. SPATIAL EFFICIENCY:")
    total_distance = 0
    for truck_id, move_ids in solution['assignment'].items():
        truck = trucks[truck_id - 1]
        current_pos = truck.start_location
        
        for move_id in move_ids:
            move = container_moves[move_id - 1]
            # Distance to origin
            total_distance += euclidean_distance(current_pos, move.origin)
            # Distance from origin to destination
            total_distance += euclidean_distance(move.origin, move.destination)
            current_pos = move.destination
    
    avg_distance_per_move = total_distance / len(container_moves)
    print(f"   Total travel distance: {total_distance:.1f} units")
    print(f"   Average distance per move: {avg_distance_per_move:.1f} units")
    print()
    
    print("5. OPERATIONAL RECOMMENDATIONS:")
    
    # Identify bottlenecks
    max_completion = max([s['completion_time'] for s in solution['move_schedule'].values()])
    bottleneck_moves = [m for m, s in solution['move_schedule'].items() if s['completion_time'] == max_completion]
    
    print(f"   • Bottleneck moves: {bottleneck_moves} (complete at {max_completion:.1f})")
    
    # Identify underutilized resources
    idle_trucks = [t for t, moves in solution['assignment'].items() if not moves]
    if idle_trucks:
        print(f"   • Underutilized trucks: {idle_trucks}")
    
    # Priority service quality
    priority1_moves = [m for m in container_moves if m.priority == 1]
    priority1_completions = [solution['move_schedule'][m.id]['completion_time'] for m in priority1_moves]
    if priority1_completions:
        print(f"   • High-priority moves complete by: {max(priority1_completions):.1f}")
    
    print(f"   • Consider rebalancing truck starting positions to reduce initial travel")
    print(f"   • Time window constraints are {'binding' if tight_windows > len(container_moves)/2 else 'non-binding'}")
else:
    print("No solution available for analysis")

## Summary

This notebook demonstrated the **Mixed-Integer Programming formulation** of the yard truck scheduling problem:

### Key achievements:
- **Exact optimization**: Found optimal schedule using enumeration for small instances
- **Comprehensive modeling**: Included time windows, priorities, and spatial constraints
- **Rich visualizations**: Gantt charts and spatial layouts for schedule understanding
- **Sensitivity analysis**: Explored impact of key parameters on solution quality

### Methodological insights:
- **Trade-offs**: Balance between priority service and resource utilization
- **Constraints**: Time windows significantly impact feasible schedules
- **Spatial efficiency**: Travel distances affect total completion times

### Next steps:
- Compare with heuristic approaches (Tier 2)
- Scale to larger instances using metaheuristics (Tier 3)
- Explore reinforcement learning for dynamic scheduling (Tier 4)

The MIP approach provides a **benchmark optimal solution** that can be used to evaluate the performance of faster, approximate methods in subsequent tiers.